In [1]:
!pip install -q openai-whisper
!pip install -q librosa
!pip install -q soundfile
!pip install -q gradio
!pip install -q nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.6 MB/s eta 0:00:00


In [3]:
import whisper
import librosa
import re
import gradio as gr
import nltk

from collections import Counter

In [4]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [5]:
print("Loading Whisper model...")

whisper_model = whisper.load_model("base")

print("Whisper model loaded successfully!")

Loading Whisper model...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 157MiB/s]


Whisper model loaded successfully!


In [6]:
def speech_to_text(audio_file):

    result = whisper_model.transcribe(
        audio_file
    )

    transcript = result["text"].strip()

    return transcript

In [7]:
meeting_keywords = [
    "project",
    "meeting",
    "deadline",
    "task",
    "team",
    "client",
    "development",
    "design",
    "testing",
    "budget",
    "schedule",
    "requirement",
    "plan",
    "problem",
    "solution",
    "discussion",
    "decision",
    "future"
]


def analyze_keywords(text):

    words = re.findall(
        r'\b\w+\b',
        text.lower()
    )

    word_frequency = Counter(words)

    found_keywords = {}

    for keyword in meeting_keywords:

        if keyword in word_frequency:

            found_keywords[keyword] = (
                word_frequency[keyword]
            )

    return found_keywords

In [8]:
def analyze_text(text):

    words = re.findall(
        r'\b\w+\b',
        text.lower()
    )

    word_count = len(words)

    sentences = re.split(
        r'[.!?]+',
        text
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    # Select first few sentences as important content
    important_sentences = sentences[:3]

    return {
        "word_count": word_count,
        "sentence_count": len(sentences),
        "important_sentences": important_sentences
    }

In [9]:
def generate_summary(text):

    sentences = re.split(
        r'[.!?]+',
        text
    )

    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    if len(sentences) == 0:
        return "No summary available."

    if len(sentences) <= 3:
        summary_sentences = sentences

    else:
        summary_sentences = sentences[:3]

    summary = ". ".join(
        summary_sentences
    )

    return summary + "."

In [10]:
def meeting_analyzer(audio_file):

    if audio_file is None:

        return (
            "Please upload or record a meeting.",
            "",
            "",
            ""
        )

    # Speech to text
    transcript = speech_to_text(
        audio_file
    )

    if not transcript:

        return (
            "Could not understand the audio.",
            "",
            "",
            ""
        )

    # Text analysis
    text_result = analyze_text(
        transcript
    )

    # Keyword analysis
    keywords = analyze_keywords(
        transcript
    )

    # Summary
    summary = generate_summary(
        transcript
    )

    # Format keywords
    if keywords:

        keyword_text = "\n".join(
            f"• {key}: {value}"
            for key, value in keywords.items()
        )

    else:

        keyword_text = "No important keywords detected."

    # Text report
    report = (
        f"Word Count: {text_result['word_count']}\n"
        f"Sentence Count: {text_result['sentence_count']}\n\n"
        f"Important Sentences:\n"
        + "\n".join(
            "• " + sentence
            for sentence in text_result[
                "important_sentences"
            ]
        )
    )

    return (
        transcript,
        summary,
        keyword_text,
        report
    )

In [11]:
with gr.Blocks(
    title="Meeting & Conversation Analyzer"
) as app:

    gr.Markdown("""
    # 📝 Meeting & Conversation Analyzer

    ### Convert meeting speech into text and generate a useful report

    **🎤 Speech → 📝 Transcript → 🔑 Keywords → 📌 Summary → 📊 Report**
    """)

    audio_input = gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="🎤 Record / Upload Meeting Audio"
    )

    analyze_button = gr.Button(
        "🔍 Analyze Meeting",
        variant="primary"
    )

    transcript_output = gr.Textbox(
        label="📝 Meeting Transcript",
        lines=8
    )

    summary_output = gr.Textbox(
        label="📌 Meeting Summary",
        lines=5
    )

    keywords_output = gr.Textbox(
        label="🔑 Important Keywords",
        lines=8
    )

    report_output = gr.Textbox(
        label="📊 Meeting Report",
        lines=10
    )

    analyze_button.click(
        fn=meeting_analyzer,
        inputs=audio_input,
        outputs=[
            transcript_output,
            summary_output,
            keywords_output,
            report_output
        ]
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://185bfe92bcdcd9961c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
